# Project 3 · Part 3: Project 3 · Part 3: City rents — a price model, error analysis and portfolio write-up · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 39

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Set the baseline

The data is prepared and split for you. Predict the median training rent for every test listing, and store the mean absolute error in dollars in baseline_mae.

<details><summary>Hint</summary>

np.full(len(y_test), y_train.median()) repeats one number; then mean_absolute_error(y_test, baseline).

</details>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))
rentals["year_built"] = rentals["year_built"].fillna(rentals["year_built"].median())
rentals["property_age"] = 2025 - rentals["year_built"]
features = ["neighbourhood", "property_type", "bedrooms", "bathrooms", "size_sqm", "furnished",
            "balcony", "parking_spaces", "pets_allowed", "property_age", "distance_to_transit_km"]
X = pd.get_dummies(rentals[features], drop_first=True, dtype=int)
y = rentals["monthly_rent_usd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = ...       # the median training rent, repeated for every test listing
baseline_mae = ...   # mean absolute error of the baseline on the test set

print(baseline_mae)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "import numpy as _np\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.linear_model import LinearRegression as _LR\nfrom sklearn.metrics import mean_absolute_error as _mae, mean_absolute_percentage_error as _mape\n_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_r[\"year_built\"] = _r[\"year_built\"].fillna(_r[\"year_built\"].median())\n_r[\"property_age\"] = 2025 - _r[\"year_built\"]\n_f = [\"neighbourhood\", \"property_type\", \"bedrooms\", \"bathrooms\", \"size_sqm\", \"furnished\", \"balcony\", \"parking_spaces\", \"pets_allowed\", \"property_age\", \"distance_to_transit_km\"]\n_X = pd.get_dummies(_r[_f], drop_first=True, dtype=int)\n_a, _b, _c, _d = _tts(_X, _r[\"monthly_rent_usd\"], test_size=0.2, random_state=42)\n_want = _mae(_d, _np.full(len(_d), _c.median()))\nassert baseline_mae is not Ellipsis and not hasattr(baseline_mae, \"__len__\"), \"baseline_mae should be one number: mean_absolute_error(y_test, baseline)\"\nassert abs(float(baseline_mae) - _want) < 1, \"baseline_mae should use the median of y_train (not y_test or the mean) as the guess for every test listing\"\n")

## Exercise 2: Linear regression on log rent

Fit the linear regression on np.log(y_train), convert its test predictions back to dollars with np.exp, and store the errors in lin_mae (dollars) and lin_pct (a share).

<details><summary>Hint</summary>

lin.fit(X_train, np.log(y_train)); lin_pred = np.exp(lin.predict(X_test)); then mean_absolute_error and mean_absolute_percentage_error.

</details>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))
rentals["year_built"] = rentals["year_built"].fillna(rentals["year_built"].median())
rentals["property_age"] = 2025 - rentals["year_built"]
features = ["neighbourhood", "property_type", "bedrooms", "bathrooms", "size_sqm", "furnished",
            "balcony", "parking_spaces", "pets_allowed", "property_age", "distance_to_transit_km"]
X = pd.get_dummies(rentals[features], drop_first=True, dtype=int)
y = rentals["monthly_rent_usd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin = LinearRegression()
# fit lin on the training features and the LOG of the training rents
lin_pred = ...   # predictions on X_test, converted back to dollars with np.exp
lin_mae = ...    # mean absolute error in dollars
lin_pct = ...    # mean absolute percentage error (a share, e.g. 0.2 = 20%)

print(lin_mae, lin_pct)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "import numpy as _np\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.linear_model import LinearRegression as _LR\nfrom sklearn.metrics import mean_absolute_error as _mae, mean_absolute_percentage_error as _mape\n_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_r[\"year_built\"] = _r[\"year_built\"].fillna(_r[\"year_built\"].median())\n_r[\"property_age\"] = 2025 - _r[\"year_built\"]\n_f = [\"neighbourhood\", \"property_type\", \"bedrooms\", \"bathrooms\", \"size_sqm\", \"furnished\", \"balcony\", \"parking_spaces\", \"pets_allowed\", \"property_age\", \"distance_to_transit_km\"]\n_X = pd.get_dummies(_r[_f], drop_first=True, dtype=int)\n_a, _b, _c, _d = _tts(_X, _r[\"monthly_rent_usd\"], test_size=0.2, random_state=42)\n_p = _np.exp(_LR().fit(_a, _np.log(_c)).predict(_b))\nassert lin_mae is not Ellipsis and not hasattr(lin_mae, \"__len__\"), \"lin_mae should be one number: mean_absolute_error(y_test, lin_pred)\"\nassert abs(float(lin_mae) - _mae(_d, _p)) < 1, \"lin_mae should be the dollar error of a model fitted on np.log(y_train), with predictions converted back by np.exp\"\nassert lin_pct is not Ellipsis and abs(float(lin_pct) - _mape(_d, _p)) < 1e-4, \"lin_pct should be mean_absolute_percentage_error(y_test, lin_pred), as a share\"\n")

## Exercise 3: Complete the valuation function

Finish the row line in estimate_rent so a described listing is encoded like the training data, then estimate the monthly rent of a 1-bed apartment in Old Town with 1 bedroom, 1 bathroom and 50 square metres.

<details><summary>Hint</summary>

pd.get_dummies(listing, dtype=int) encodes the text columns; .reindex(columns=X.columns, fill_value=0) adds the missing dummy columns in training order.

</details>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))
rentals["year_built"] = rentals["year_built"].fillna(rentals["year_built"].median())
rentals["property_age"] = 2025 - rentals["year_built"]
features = ["neighbourhood", "property_type", "bedrooms", "bathrooms", "size_sqm", "furnished",
            "balcony", "parking_spaces", "pets_allowed", "property_age", "distance_to_transit_km"]
X = pd.get_dummies(rentals[features], drop_first=True, dtype=int)
y = rentals["monthly_rent_usd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin = LinearRegression().fit(X_train, np.log(y_train))

def estimate_rent(neighbourhood, property_type, bedrooms, bathrooms, size_sqm, furnished=0, balcony=0,
                  parking_spaces=0, pets_allowed=0, property_age=30, distance_to_transit_km=0.5):
    listing = pd.DataFrame([{
        "neighbourhood": neighbourhood, "property_type": property_type, "bedrooms": bedrooms,
        "bathrooms": bathrooms, "size_sqm": size_sqm, "furnished": furnished, "balcony": balcony,
        "parking_spaces": parking_spaces, "pets_allowed": pets_allowed, "property_age": property_age,
        "distance_to_transit_km": distance_to_transit_km}])
    row = ...   # one-hot encode listing, then reindex to X.columns, filling missing columns with 0
    return float(np.exp(lin.predict(row)[0]))

old_town_flat = ...   # a 1-bed apartment in Old Town: 1 bedroom, 1 bathroom, 50 sqm (other details default)
print(old_town_flat)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "import numpy as _np\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.linear_model import LinearRegression as _LR\nfrom sklearn.metrics import mean_absolute_error as _mae, mean_absolute_percentage_error as _mape\n_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_r[\"year_built\"] = _r[\"year_built\"].fillna(_r[\"year_built\"].median())\n_r[\"property_age\"] = 2025 - _r[\"year_built\"]\n_f = [\"neighbourhood\", \"property_type\", \"bedrooms\", \"bathrooms\", \"size_sqm\", \"furnished\", \"balcony\", \"parking_spaces\", \"pets_allowed\", \"property_age\", \"distance_to_transit_km\"]\n_X = pd.get_dummies(_r[_f], drop_first=True, dtype=int)\n_a, _b, _c, _d = _tts(_X, _r[\"monthly_rent_usd\"], test_size=0.2, random_state=42)\n_m = _LR().fit(_a, _np.log(_c))\n_row = pd.DataFrame([dict(zip(_X.columns, [0.0] * _X.shape[1]))])\n_row.loc[0, [\"bedrooms\", \"bathrooms\", \"size_sqm\", \"property_age\", \"distance_to_transit_km\"]] = [1, 1, 50, 30, 0.5]\n_row.loc[0, \"neighbourhood_Old Town\"] = 1\n_want = float(_np.exp(_m.predict(_row)[0]))\nassert old_town_flat is not Ellipsis and not hasattr(old_town_flat, \"__len__\"), \"old_town_flat should be one number: estimate_rent(\\\"Old Town\\\", \\\"1-bed apartment\\\", 1, 1, 50)\"\nassert abs(float(old_town_flat) - _want) / _want < 1e-6, \"Check row: pd.get_dummies(listing, dtype=int).reindex(columns=X.columns, fill_value=0), and call estimate_rent(\\\"Old Town\\\", \\\"1-bed apartment\\\", 1, 1, 50)\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=39) and take the quiz.